In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## Dataverwerking

In [2]:
YEAR = 'year'
COUNTRY = 'country'
CATEGORY = 'category'
PERCENTAGE = 'percentage'

In [3]:
selected_year = str(2014)

In [4]:
# Defining paths
relative_root = "./../../assets/nato-defence-expenditure"

exp_per_capita_path = f"{relative_root}/spenditurepercapita2021usdollars.csv"
real_change_path = f"{relative_root}/realchange.csv"

exp_per_category_paths = {
    category: f"{relative_root}/portionofexpenditure_{category}.csv"
    for category in ['equipment', 'infrastructure', 'personel', 'other']
}

In [5]:
# Reading sources

exp_per_capita_raw = pd.read_csv(f"{exp_per_capita_path}")
real_change_raw = pd.read_csv(f"{real_change_path}")

exp_per_category_raw = {
    category: pd.read_csv(file)
    for category, file in exp_per_category_paths.items()
}

# Mapping and conversions

exp_per_capita_years = sorted([year for year in exp_per_capita_raw.columns if year.isdigit()])
exp_per_capita = exp_per_capita_raw.set_index(COUNTRY)[exp_per_capita_years]

real_change_years = sorted([year for year in real_change_raw.columns if year.isdigit()])
real_change = real_change_raw.set_index(COUNTRY)[real_change_years]

# Map expenditures to categories
exp_per_category = pd.concat(
    [df.assign(category=expense_category) for expense_category, df in exp_per_category_raw.items()],
    ignore_index=True
)

# Mapping expenditure year columns to rows
exp_per_category_years = sorted([year for year in exp_per_category.columns if year.isdigit()])
exp_per_category = exp_per_category.melt(
    id_vars=[COUNTRY, CATEGORY],
    value_vars=exp_per_category_years,
    var_name=YEAR,
    value_name=PERCENTAGE
)

# Show datasets individually, for exploration purposes

KeyError: "None of ['country'] are in the columns"

In [ ]:
exp_per_capita

In [ ]:
real_change

In [ ]:
exp_per_category

## Visualisaties

In [ ]:
# Voor subfiguren
subfig_n_cols = 4

# Voor plots
pie_start_angle = 90

Hoe veel geeft elk individueel land uit aan elke categorie?

Categorieën:
- Equipment
- Infrastructure
- Personnel
- Other

In [ ]:
selected_year_exp_per_category = (exp_per_category[exp_per_category[YEAR] == selected_year]
                                  .copy()
                                  .pivot(index=COUNTRY, columns=CATEGORY, values=PERCENTAGE)
                                  )

# Determine subfigure sizes
n_subfigs = len(selected_year_exp_per_category)
subfig_n_rows = math.ceil(n_subfigs / subfig_n_cols)

fig, axes = plt.subplots(subfig_n_rows, subfig_n_cols, figsize=(subfig_n_cols * 3, subfig_n_rows * 3))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for index, (country, row) in enumerate(selected_year_exp_per_category.iterrows()):
    axis = axes[index]
    categories, percentages = row.index, row.values
    axis.pie(percentages, labels=categories, autopct='%1.0f%%', startangle=pie_start_angle)
    axis.set_title(f"{country} - {selected_year}")
    axis.margins(0)
    axis.axis('equal')

# Hide any unused subplots
for axis in axes[n_subfigs:]:
    axis.axis('off')

plt.tight_layout()
plt.show()

Hoe veel geeft elk land uit doorheen de tijd?

In [ ]:
# Determine fixed max y
exp_per_capita_maxy = max(exp_per_capita.max())

# Plots
n_subfigs = len(exp_per_capita)
subfig_n_rows = math.ceil(n_subfigs / subfig_n_cols)

fig, axes = plt.subplots(subfig_n_rows, subfig_n_cols, figsize=(subfig_n_cols * 3, subfig_n_rows * 3))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for index, (country, row) in enumerate(exp_per_capita.iterrows()):
    axis = axes[index]
    exps = row[exp_per_capita_years]

    axis.barh(exp_per_capita_years, exps)
    axis.set_title(country)
    axis.set_ylabel("Year")
    axis.set_xlabel("Exp per capita US$ 2021")
    axis.set_xlim([0, exp_per_capita_maxy])

# Hide any unused subplots
for axis in axes[n_subfigs:]:
    axis.axis('off')

plt.tight_layout()
plt.show()

Hoe veel verandert de uitgave doorheen de jaren per land?

In [ ]:
# Determine fixed y-range for all subplots
global_min = float("inf")
global_max = float("-inf")

for _, row in real_change.iterrows():
    start = 100.0
    country_min = start
    country_max = start

    for change in row[real_change_years]:
        end = start * (1 + change / 100.0)
        low, high = sorted((start, end))
        country_min = min(country_min, low)
        country_max = max(country_max, high)
        start = end

    global_min = min(global_min, country_min)
    global_max = max(global_max, country_max)

# Plots
n_subfigs = len(real_change)
subfig_n_rows = math.ceil(n_subfigs / subfig_n_cols)

fig, axes = plt.subplots(subfig_n_rows, subfig_n_cols, figsize=(subfig_n_cols * 3, subfig_n_rows * 3))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for index, (country, row) in enumerate(real_change.iterrows()):
    axis = axes[index]

    start = 100
    starts, heights, colors = [], [], []

    for change in row[real_change_years]:
        end = start * (1 + change / 100.0)
        starts.append(start)
        heights.append(end - start)
        colors.append('green' if change > 0 else 'red')
        start = end

    axis.bar(real_change_years, heights, bottom=starts, color=colors)
    axis.set_title(country)
    axis.set_xlabel("Year")
    axis.set_ylabel("Real change %")
    axis.axhline(y=0, color='black', linewidth=0.5)
    axis.set_ylim(global_min, global_max)
    axis.tick_params(axis='both')

# Hide any unused subplots
for axis in axes[n_subfigs:]:
    axis.axis('off')

plt.tight_layout()
plt.show()